## Thư viện

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/baobaocin/data-zalo/Data_Zalo (3).csv


In [2]:
!pip install sentence-transformers==2.7.0 transformers==4.40.0 tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0
  Attempting uninstall: sentence-transformers
    Found existing inst

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sentence_transformers import InputExample
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [4]:
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers import evaluation

## Xử lý data

In [5]:
# Đọc data
df = pd.read_csv("/kaggle/input/datasets/baobaocin/data-zalo/Data_Zalo (3).csv")

# Chia data
train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train : {len(train_df):,} samples")
print(f"Val   : {len(val_df):,} samples")

Train : 31,331 samples
Val   : 1,649 samples


In [6]:
def build_examples(df):
    examples = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        query    = "query: "   + str(row["query"])
        positive = "passage: " + str(row["positive"])
        hard_neg = "passage: " + str(row["hard_neg"])

        examples.append(InputExample(texts=[query, positive, hard_neg]))

    return examples

# Build examples theo cấu trúc của model được finetune
train_examples = build_examples(train_df)

# Wrap vào DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

print(f"Training examples : {len(train_examples):,}")
print(f"Batches per epoch : {len(train_dataloader):,}")

  0%|          | 0/31331 [00:00<?, ?it/s]

Training examples : 31,331
Batches per epoch : 3,917


## Xử lý model

### Load model

In [7]:
# Load model
model = SentenceTransformer("intfloat/multilingual-e5-base")
model.max_seq_length = 512

print(f"Model loaded!")
print(f"Embedding dim : {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Model loaded!
Embedding dim : 768


### Khởi tạo loss

In [8]:
loss_fn = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.5
)

print("Loss function: TripletLoss")

Loss function: TripletLoss


### Hàm theo dõi và đánh giá cho tửng epoch

In [9]:
def build_evaluator(df, name="val"):
    sentences1, sentences2, scores = [], [], []

    for _, row in df.iterrows():
        query    = "query: "   + str(row["query"])
        positive = "passage: " + str(row["positive"])
        hard_neg = "passage: " + str(row["hard_neg"])

        sentences1.extend([query,    query])
        sentences2.extend([positive, hard_neg])
        scores.extend([1.0, 0.0])

    return evaluation.EmbeddingSimilarityEvaluator(
        sentences1=sentences1,
        sentences2=sentences2,
        scores=scores,
        name=name,
        write_csv=True,
    )

val_evaluator = build_evaluator(val_df, name="val")
print(f"Evaluator built!")

Evaluator built!


In [10]:
# Tính warmup steps
total_steps  = len(train_dataloader) * 5  # 5 epochs
warmup_steps = int(total_steps * 0.1)     # 10% đầu để warm up

print(f"Total steps  : {total_steps:,}")
print(f"Warmup steps : {warmup_steps:,}")

# Bắt đầu train
model.fit(
    train_objectives=[(train_dataloader, loss_fn)],
    evaluator=val_evaluator,
    epochs=5,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": 2e-5},
    output_path="/kaggle/working/multilingual-e5-base-legal-v2",
    save_best_model=True,
    use_amp=True,        # FP16, tắt đi nếu không có GPU
    show_progress_bar=True,
)

print("Training hoàn tất!")

Total steps  : 19,585
Warmup steps : 1,958


/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:1011: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3917 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3917 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3917 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3917 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3917 [00:00<?, ?it/s]

Training hoàn tất!
